In [2]:
import networkx as nx
import numpy as np

def ncssi(
    G1,
    G2,
    bidirectional1=False,
    bidirectional2=False,
    community_attr='community',
    weight_attr='weight'
):
    """
    Compute the Network Community Structure Similarity Index (NCSSI) between two graphs.

    Parameters
    ----------
    G1 : NetworkX graph
        The first graph.
    G2 : NetworkX graph
        The second graph.
    bidirectional1 : bool, optional (default=False)
        If True, treat G1 as bidirectional (undirected).
    bidirectional2 : bool, optional (default=False)
        If True, treat G2 as bidirectional (undirected).
    community_attr : str, optional (default='community')
        The node attribute name that stores the community information.
    weight_attr : str, optional (default='weight')
        The edge attribute name that stores the weight information.

    Returns
    -------
    float
        The NCSSI value between G1 and G2.

    Raises
    ------
    TypeError
        If the inputs are not of the expected types.
    ValueError
        If required attributes are missing or incorrectly formatted.

    Notes
    -----
    The NCSSI is a measure of similarity between two networks considering their community structures,
    edge weights, and community labels. It ranges from 0 to 1, where 1 indicates identical community
    structures, and 0 indicates completely dissimilar structures.
    """
    # Validate input types
    if not isinstance(G1, nx.Graph):
        raise TypeError("G1 must be a NetworkX graph.")
    if not isinstance(G2, nx.Graph):
        raise TypeError("G2 must be a NetworkX graph.")
    if not isinstance(bidirectional1, bool):
        raise TypeError("bidirectional1 must be a boolean.")
    if not isinstance(bidirectional2, bool):
        raise TypeError("bidirectional2 must be a boolean.")
    if not isinstance(community_attr, str):
        raise TypeError("community_attr must be a string.")
    if not isinstance(weight_attr, str):
        raise TypeError("weight_attr must be a string.")

    # Copy graphs to avoid modifying the original ones
    G1 = G1.copy()
    G2 = G2.copy()

    # Prepare the graphs: get node-community mappings, edge weights, and communities
    node_to_community_G1, edge_weights_G1, communities_G1 = _prepare_graph(
        G1, bidirectional1, community_attr, weight_attr
    )
    node_to_community_G2, edge_weights_G2, communities_G2 = _prepare_graph(
        G2, bidirectional2, community_attr, weight_attr
    )

    # Map node labels to indices for internal processing
    try:
        all_nodes = set(node_to_community_G1.keys()).union(set(node_to_community_G2.keys()))
        node_label_mapping = {node: idx for idx, node in enumerate(all_nodes)}
    except Exception as e:
        raise ValueError(f"Error mapping node labels to indices: {e}")

    # Remap node labels in edge weights and communities
    edge_weights_G1 = {
        (node_label_mapping[u], node_label_mapping[v]): w
        for (u, v), w in edge_weights_G1.items()
    }
    edge_weights_G2 = {
        (node_label_mapping[u], node_label_mapping[v]): w
        for (u, v), w in edge_weights_G2.items()
    }
    node_to_community_G1 = {
        node_label_mapping[node]: com for node, com in node_to_community_G1.items()
    }
    node_to_community_G2 = {
        node_label_mapping[node]: com for node, com in node_to_community_G2.items()
    }
    communities_G1 = {
        com: set(node_label_mapping[node] for node in nodes)
        for com, nodes in communities_G1.items()
    }
    communities_G2 = {
        com: set(node_label_mapping[node] for node in nodes)
        for com, nodes in communities_G2.items()
    }

    # Find the best matching between communities
    community_pairs = _match_communities(
        communities_G1, communities_G2, edge_weights_G1, edge_weights_G2
    )

    # Get all node indices
    all_nodes_indices = set(node_to_community_G1.keys()).union(set(node_to_community_G2.keys()))

    # Initialize variables to accumulate adjusted edit costs and adjustment factors
    total_adjusted_edit_cost_G1 = 0.0
    total_adjustment_factor_G1 = 0.0
    total_adjusted_edit_cost_G2 = 0.0
    total_adjustment_factor_G2 = 0.0

    for node in all_nodes_indices:
        # Get community labels for the node in both graphs (-1 if not present)
        community_label_G1 = node_to_community_G1.get(node, -1)
        community_label_G2 = node_to_community_G2.get(node, -1)

        # Get the matched community labels
        matched_community_label_G2 = community_pairs.get(community_label_G1, -1)
        inverse_pairs = {v: k for k, v in community_pairs.items()}
        matched_community_label_G1 = inverse_pairs.get(community_label_G2, -1)

        # Initialize temporary variables for edit costs and adjustment factors
        edit_cost_G1, adjustment_factor_G1 = 0.0, 0.0
        edit_cost_G2, adjustment_factor_G2 = 0.0, 0.0

        # Compute edit cost and adjustment factor for G1
        if community_label_G1 != -1:
            # Nodes in the community of node in G1
            community_nodes_G1 = communities_G1[community_label_G1]
            # Sum of weights for intra-community edges
            adjustment_factor_G1 += sum(
                edge_weights_G1.get((node, neighbor), 0.0)
                for neighbor in community_nodes_G1
            )
            # If the communities are different, compute the edit cost
            if (
                matched_community_label_G1 != -1
                and matched_community_label_G1 != community_label_G1
            ):
                matched_community_nodes_G1 = communities_G1[matched_community_label_G1]
                edit_cost_G1 += sum(
                    edge_weights_G1.get((node, neighbor), 0.0)
                    for neighbor in matched_community_nodes_G1
                )
                adjustment_factor_G1 += abs(edit_cost_G1)

        # Compute edit cost and adjustment factor for G2
        if community_label_G2 != -1:
            community_nodes_G2 = communities_G2[community_label_G2]
            adjustment_factor_G2 += sum(
                edge_weights_G2.get((node, neighbor), 0.0)
                for neighbor in community_nodes_G2
            )
            if (
                matched_community_label_G2 != -1
                and matched_community_label_G2 != community_label_G2
            ):
                matched_community_nodes_G2 = communities_G2[matched_community_label_G2]
                edit_cost_G2 += sum(
                    edge_weights_G2.get((node, neighbor), 0.0)
                    for neighbor in matched_community_nodes_G2
                )
                adjustment_factor_G2 += abs(edit_cost_G2)

        # Accumulate adjusted edit costs and adjustment factors
        total_adjusted_edit_cost_G1 += abs(edit_cost_G1)
        total_adjustment_factor_G1 += abs(adjustment_factor_G1)
        total_adjusted_edit_cost_G2 += abs(edit_cost_G2)
        total_adjustment_factor_G2 += abs(adjustment_factor_G2)

    # Compute the adjusted edit cost ratios
    adjusted_edit_cost_ratio_G1 = (
        total_adjusted_edit_cost_G1 / total_adjustment_factor_G1
        if total_adjustment_factor_G1
        else 0
    )
    adjusted_edit_cost_ratio_G2 = (
        total_adjusted_edit_cost_G2 / total_adjustment_factor_G2
        if total_adjustment_factor_G2
        else 0
    )

    # Compute final NCSSI value
    ncssi_value = 1 - ((adjusted_edit_cost_ratio_G1 + adjusted_edit_cost_ratio_G2) / 2)
    return ncssi_value

def _prepare_graph(G, bidirectional, community_attr, weight_attr):
    """
    Prepare the graph structure for NCSSI computation.

    Parameters
    ----------
    G : NetworkX graph
        The graph to process.
    bidirectional : bool
        If True, treat G as bidirectional (undirected).
    community_attr : str
        The node attribute name that stores the community information.
    weight_attr : str
        The edge attribute name that stores the weight information.

    Returns
    -------
    node_to_community : dict
        Mapping from node to community label.
    edge_weights : dict
        Edge weight dictionary {(u, v): weight}.
    communities : dict
        Mapping from community label to set of nodes.

    Raises
    ------
    ValueError
        If required attributes are missing or incorrectly formatted.
    """
    # Extract the set of communities
    try:
        community_set = set(
            frozenset(G.nodes[node][community_attr]) for node in G if community_attr in G.nodes[node]
        )
    except Exception as e:
        raise ValueError(f"Error extracting communities: {e}")

    if not community_set:
        raise ValueError(f"No community information found using attribute '{community_attr}'.")

    community_list = list(community_set)
    community_label_map = {community: idx for idx, community in enumerate(community_list)}

    # Map nodes to community labels
    node_to_community = {}
    for node in G:
        if community_attr not in G.nodes[node]:
            raise ValueError(f"Node '{node}' does not have the community attribute '{community_attr}'.")
        community_attr_value = G.nodes[node][community_attr]
        if not isinstance(community_attr_value, (list, set, frozenset)):
            raise ValueError(
                f"Community attribute for node '{node}' must be a list or set, got {type(community_attr_value)}."
            )
        matched = False
        for idx, community in enumerate(community_list):
            if set(community_attr_value) == set(community):
                node_to_community[node] = idx
                matched = True
                break
        if not matched:
            raise ValueError(f"Community for node '{node}' does not match any known communities.")

    # Map community labels to nodes
    communities = {idx: set() for idx in range(len(community_list))}
    for node, com_label in node_to_community.items():
        communities[com_label].add(node)

    # Create edge weight dictionary
    edge_weights = {}
    for u, v, data in G.edges(data=True):
        if weight_attr in data:
            weight = data[weight_attr]
            if not isinstance(weight, (int, float)):
                raise ValueError(
                    f"Edge weight for edge ({u}, {v}) must be a number, got {type(weight)}."
                )
        else:
            weight = 1.0  # Default weight
        edge_weights[(u, v)] = weight
        if not bidirectional:
            edge_weights[(v, u)] = weight

    return node_to_community, edge_weights, communities

def _compute_overlap_score(nodes_a, nodes_b, edge_weights_G1, edge_weights_G2):
    """
    Compute the Edge-Based Overlapping Score between two communities.

    Parameters
    ----------
    nodes_a : set
        Nodes in community A.
    nodes_b : set
        Nodes in community B.
    edge_weights_G1 : dict
        Edge weight dictionary for the first graph.
    edge_weights_G2 : dict
        Edge weight dictionary for the second graph.

    Returns
    -------
    float
        The Edge-Based Overlapping Score between communities A and B.
    """
    intersection_nodes = nodes_a & nodes_b

    # Sum of edge weights in the intersection
    sum_edges_intersection = sum(
        edge_weights_G1.get((i, j), 0.0) + edge_weights_G2.get((i, j), 0.0)
        for i in intersection_nodes for j in intersection_nodes
    )

    # Sum of edge weights in the union
    sum_edges_union = (
        sum(edge_weights_G1.get((i, j), 0.0) for i in nodes_a for j in nodes_a) +
        sum(edge_weights_G2.get((i, j), 0.0) for i in nodes_b for j in nodes_b)
    )

    return sum_edges_intersection / sum_edges_union if sum_edges_union else 0.0

def _match_communities(communities_G1, communities_G2, edge_weights_G1, edge_weights_G2):
    """
    Find the best matching between communities in two graphs.

    Parameters
    ----------
    communities_G1 : dict
        Communities in the first graph {label: set of nodes}.
    communities_G2 : dict
        Communities in the second graph {label: set of nodes}.
    edge_weights_G1 : dict
        Edge weight dictionary for the first graph.
    edge_weights_G2 : dict
        Edge weight dictionary for the second graph.

    Returns
    -------
    dict
        Mapping from community labels in communities_G1 to labels in communities_G2.

    Raises
    ------
    ValueError
        If there is an error during community matching.
    """
    labels_G1 = list(communities_G1.keys())
    labels_G2 = list(communities_G2.keys())

    # Initialize the overlap score matrix
    overlap_scores = np.zeros((len(labels_G1), len(labels_G2)))

    # Normalize edge weights
    total_weight_G1 = sum(edge_weights_G1.values())
    total_weight_G2 = sum(edge_weights_G2.values())

    edge_weights_G1_norm = {}
    if total_weight_G1:
        for edge, weight in edge_weights_G1.items():
            if not isinstance(weight, (int, float)):
                raise ValueError(f"Edge weight must be numeric, got {type(weight)} for edge {edge}.")
            edge_weights_G1_norm[edge] = weight / total_weight_G1
    else:
        edge_weights_G1_norm = edge_weights_G1

    edge_weights_G2_norm = {}
    if total_weight_G2:
        for edge, weight in edge_weights_G2.items():
            if not isinstance(weight, (int, float)):
                raise ValueError(f"Edge weight must be numeric, got {type(weight)} for edge {edge}.")
            edge_weights_G2_norm[edge] = weight / total_weight_G2
    else:
        edge_weights_G2_norm = edge_weights_G2

    # Compute overlap scores for all pairs of communities
    for i, label_a in enumerate(labels_G1):
        for j, label_b in enumerate(labels_G2):
            nodes_a = communities_G1[label_a]
            nodes_b = communities_G2[label_b]
            try:
                score = _compute_overlap_score(nodes_a, nodes_b, edge_weights_G1_norm, edge_weights_G2_norm)
            except Exception as e:
                raise ValueError(f"Error computing overlap score between communities {label_a} and {label_b}: {e}")
            overlap_scores[i, j] = score

    # Find the best matches based on the overlap scores
    community_pairs = {}
    try:
        while overlap_scores.size and np.max(overlap_scores) > 0:
            idx = np.unravel_index(np.argmax(overlap_scores), overlap_scores.shape)
            label_a = labels_G1[idx[0]]
            label_b = labels_G2[idx[1]]
            community_pairs[label_a] = label_b
            overlap_scores[idx[0], :] = -1  # Exclude this community in future matches
            overlap_scores[:, idx[1]] = -1
    except Exception as e:
        raise ValueError(f"Error during community matching: {e}")

    return community_pairs
